# V2 Synthetic Positive Control (Train-Domain, Semi-Synthetic Labels)

**Designation (verbatim, in every artifact and summary):** train-domain synthetic
positive control (semi-synthetic labels, protocol-validation evidence only).

**Research question.** Does the frozen evaluation protocol's readout respond
monotonically to a KNOWN planted signal of controlled strength, and does it read
null when the planted strength is zero? This exercises the protocol's
measurement chain (same-row stratified-dummy floor, macro-F1 delta, per-ticker
positivity, train-inner folds, frozen seeds, label-shuffle sentinel) on labels
whose ground truth is known by construction.

**INVARIANT (bold, enforced in code):** the entire experiment runs on the frozen
TRAIN segment only (1998-01-02 through 2013-09-16, end-exclusive). ZERO contact
with the official validation split. ZERO contact with post-2017 rows. The stage
code raises on any timestamp at or after 2013-09-16.

**Evidence status.** Every number this notebook produces is synthetic-label
protocol-validation evidence about the measurement chain. It is NEVER market
evidence, never model evidence, and never enters a model-performance claim.

Preregistration (frozen before any arm runs):
`docs/protocols/v2_positive_control_preregistration_20260701.md`


## Preregistration Summary

- Injection: for each eligible train row, the binary label is relabeled to agree
  with a fixed, feature-measurable rule (sign of the target bar's day-local
  `log_return`, the last bar of every input window) with probability 0.5 + s.
  Eligibility, features, timestamps, and validity flags are untouched; only the
  label changes. s = 0 makes labels independent fair coins (a true null).
- Arms: s in {0.000 (mandatory null), 0.010 (observed-edge scale), 0.020
  (must-detect), 0.050 (must-detect ceiling)}; one frozen synthetic label set
  per arm (injection seed 20260701, per-row sha256 uniforms).
- Machinery: identical to the executed Stage 02 train-inner path — same Stage 00
  frozen artifacts, same feature/window rebuild, 3 chronological train-inner
  folds, frozen seeds [101, 202], same fold-row caps, the four registry
  baselines recomputed on the synthetic labels per fold/seed, and the
  predeclared primary configuration only (`tcn_tiny` profile `tcn_p01` on
  candidate `price_volume_time_w20`).
- Predeclared criteria (prereg section 6): P1 the s=0 arm must read within the
  null band (train-inner control spread of the identical real-data rows) with a
  non-positive LCB; P2 mean deltas strictly increase across {0, 0.02, 0.05};
  P3 the 0.02 and 0.05 arms must flag a signal (positive mean, positive LCB,
  per-ticker floor). The 0.01 arm is reported, never gated.
- Diagnostics per arm (prereg section 7): dummy-floor flatness, per-ticker
  positivity, and the label-shuffle / time-reverse sentinels (the planted edge
  is a row-level within-day coupling, so the shuffle is expected to collapse it).

FORBIDDEN wording in anything built on this run: market evidence, real edge
confirmed, model evidence, profitable, clean test, final model.


## Expected Artifacts

```
/content/lst_models_results/v2_synthetic_positive_control/<run_id>/
  run_manifest.json
  artifact_inventory.csv
  spc_trial_ledger.csv
  spc_arm_summary.csv
  spc_baseline_control_summary.csv
  spc_sentinel_ledger.csv
  spc_injection_manifest.json
  spc_criteria_readout.json
  spc_trials_arm_s0p000.csv
  spc_trials_arm_s0p010.csv
  spc_trials_arm_s0p020.csv
  spc_trials_arm_s0p050.csv
```

Durable backup: `My Drive/lst_models/results/v2_synthetic_positive_control/<run_id>/`


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import subprocess
import sys
import zipfile

RUN_PROJECT_BOOTSTRAP = True
PROJECT_BOOTSTRAP_MODE = "github_commit"  # github_commit | manual_upload | already_present

PROJECT_REPO_URL = "https://github.com/zkc768/lstm-zhang.git"
# Two-step exact-commit pin: push the full v2_synthetic_positive_control bundle
# (notebook + config + protocol + src + tests) first, then fill
# PROJECT_REPO_COMMIT with the v2 synthetic positive control full-bundle commit.
PROJECT_REPO_COMMIT = "1ec06811f427f31bd6751b6bb9f7cf2c2ce123a7"
PROJECT_ROOT = Path("/content/lst_models")

RUN_STAGE00_DRIVE_SYNC = True
RUN_STAGE01_DRIVE_SYNC = True
RUN_STAGE02_REAL_DRIVE_SYNC = True
RUN_RAW_DATA_SYNC = True
RUN_SPC = False
RUN_SPC_DRIVE_BACKUP = True
RUN_ARTIFACT_DISPLAY = False

STAGE_NAME = "v2_synthetic_positive_control"
ROUTE = "lst_models"
SCOPE = "validation_only"
HOLDOUT_TEST_CONTACT = False
SYNTHETIC_LABELS = True
TRAIN_DOMAIN_ONLY = True
TRAIN_END_EXCLUSIVE = "2013-09-16"
FROZEN_SEEDS = [101, 202]
ARM_STRENGTHS = [0.0, 0.01, 0.02, 0.05]
INJECTION_SEED = 20260701
STAGE00_RUN_ID = "20260610_051705_347450"
STAGE01_RUN_ID = "20260610_075002"
STAGE02_REAL_RUN_ID = "20260610_082130_797479"
SPC_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
STAGE00_OUTPUT_DIR = Path("/content/lst_models_results/00_data_split_label_freeze") / STAGE00_RUN_ID
STAGE01_OUTPUT_DIR = Path("/content/lst_models_results/01_feature_window_search") / STAGE01_RUN_ID
STAGE02_REAL_OUTPUT_DIR = Path("/content/lst_models_results/02_model_hpo_train_inner") / STAGE02_REAL_RUN_ID
OUTPUT_DIR = Path("/content/lst_models_results/v2_synthetic_positive_control")
SPC_OUTPUT_DIR = OUTPUT_DIR / SPC_RUN_ID
STAGE00_DRIVE_PATH_PARTS = ["lst_models", "results", "00_data_split_label_freeze", STAGE00_RUN_ID]
STAGE01_DRIVE_PATH_PARTS = ["lst_models", "results", "01_feature_window_search", STAGE01_RUN_ID]
STAGE02_REAL_DRIVE_PATH_PARTS = ["lst_models", "results", "02_model_hpo_train_inner", STAGE02_REAL_RUN_ID]
SPC_DRIVE_RESULT_PATH_PARTS = ["lst_models", "results", "v2_synthetic_positive_control"]
RAW_DATA_DIR = Path("/content/lst_models_raw_stock_data")
CHECKPOINT_ROOT = Path("/content/lst_models_checkpoints/v2_synthetic_positive_control")

def quote_drive_query_value(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")

def run_cmd(args, cwd=None):
    print("+", " ".join(str(arg) for arg in args))
    subprocess.run(args, cwd=cwd, check=True)

def looks_like_project_root(path):
    return (
        (path / "configs" / "stages" / "v2_synthetic_positive_control.yaml").exists()
        and (path / "docs" / "protocols" / "v2_positive_control_preregistration_20260701.md").exists()
        and (path / "notebooks" / "v2_synthetic_positive_control_colab.ipynb").exists()
        and (path / "src" / "lst_models" / "stages" / "synthetic_positive_control.py").exists()
        and (path / "src" / "lst_models" / "synthetic_control.py").exists()
    )

def safe_extract_project_zip(zip_path):
    destination = Path("/content").resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
            target = (destination / member_path).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
        archive.extractall(destination)
    for candidate in [Path("/content/lst_models"), Path("/content") / zip_path.stem]:
        if looks_like_project_root(candidate):
            return candidate
    raise FileNotFoundError("No lst_models project root found after zip extraction.")

def upload_and_extract_project_zip():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("manual_upload mode only works inside Colab.") from exc
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith(".zip")]
    if not zip_names:
        raise ValueError("Upload a .zip file containing the lst_models project folder.")
    return safe_extract_project_zip(Path("/content") / zip_names[0])

if RUN_PROJECT_BOOTSTRAP:
    if PROJECT_BOOTSTRAP_MODE == "github_commit":
        if "REPLACE_WITH" in PROJECT_REPO_COMMIT:
            raise ValueError(
                "Fill PROJECT_REPO_COMMIT with the v2 synthetic positive control "
                "full-bundle commit before running (two-step exact-commit pin)."
            )
        if (PROJECT_ROOT / ".git").exists():
            run_cmd(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        else:
            run_cmd(["git", "clone", PROJECT_REPO_URL, str(PROJECT_ROOT)])
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
        assert actual_commit == PROJECT_REPO_COMMIT, (actual_commit, PROJECT_REPO_COMMIT)
    elif PROJECT_BOOTSTRAP_MODE == "manual_upload":
        PROJECT_ROOT = upload_and_extract_project_zip()
    elif PROJECT_BOOTSTRAP_MODE == "already_present":
        pass
    else:
        raise ValueError("PROJECT_BOOTSTRAP_MODE must be github_commit, manual_upload, or already_present")

STAGE_CONFIG_PATH = PROJECT_ROOT / "configs" / "stages" / "v2_synthetic_positive_control.yaml"
PROTOCOL_PATH = PROJECT_ROOT / "docs" / "protocols" / "v2_positive_control_preregistration_20260701.md"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "v2_synthetic_positive_control_colab.ipynb"
STAGE_ENTRYPOINT_PATH = PROJECT_ROOT / "src" / "lst_models" / "stages" / "synthetic_positive_control.py"
DOMAIN_MODULE_PATH = PROJECT_ROOT / "src" / "lst_models" / "synthetic_control.py"
RAW_DATA_MANIFEST_PATH = PROJECT_ROOT / "configs" / "lst_models_data.yaml"
TCN_SEARCH_SPACE_PATH = PROJECT_ROOT / "configs" / "models" / "tcn" / "search_space.yaml"
REQUIRED_PROJECT_FILES = [
    STAGE_CONFIG_PATH, PROTOCOL_PATH, NOTEBOOK_PATH, STAGE_ENTRYPOINT_PATH,
    DOMAIN_MODULE_PATH, RAW_DATA_MANIFEST_PATH, TCN_SEARCH_SPACE_PATH,
]
missing_project_files = [path for path in REQUIRED_PROJECT_FILES if not path.exists()]
if missing_project_files:
    missing_text = "\n".join(f"- {path}" for path in missing_project_files)
    raise FileNotFoundError("v2 synthetic positive control bootstrap target is missing required files:\n" + missing_text)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPC_RUN_ID:", SPC_RUN_ID)


## Config Load And Contract Check


In [ ]:
try:
    import yaml
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("PyYAML is required to read the v2 synthetic positive control config.") from exc

def require_path(path):
    if not path.exists():
        raise FileNotFoundError(f"missing required v2 synthetic positive control file: {path}")

require_path(STAGE_CONFIG_PATH)
require_path(PROTOCOL_PATH)
require_path(RAW_DATA_MANIFEST_PATH)

with STAGE_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    spc_config = yaml.safe_load(handle)

stage_inputs = spc_config["inputs"]
stage_outputs = spc_config["outputs"]
stage_checkpointing = spc_config["checkpointing"]
stage_inputs["stage00_run_id"] = STAGE00_RUN_ID
stage_inputs["stage00_runtime_run_dir"] = str(STAGE00_OUTPUT_DIR)
stage_inputs["stage00_drive_path_parts"] = STAGE00_DRIVE_PATH_PARTS
stage_inputs["stage01_run_id"] = STAGE01_RUN_ID
stage_inputs["stage01_runtime_run_dir"] = str(STAGE01_OUTPUT_DIR)
stage_inputs["stage01_drive_path_parts"] = STAGE01_DRIVE_PATH_PARTS
stage_inputs["stage02_real_run_id"] = STAGE02_REAL_RUN_ID
stage_inputs["stage02_real_runtime_run_dir"] = str(STAGE02_REAL_OUTPUT_DIR)
stage_inputs["stage02_real_drive_path_parts"] = STAGE02_REAL_DRIVE_PATH_PARTS
stage_inputs["raw_data_dir"] = str(RAW_DATA_DIR)
stage_outputs["output_dir"] = str(OUTPUT_DIR)
stage_outputs["run_id"] = SPC_RUN_ID
stage_checkpointing["checkpoint_dir"] = str(CHECKPOINT_ROOT)

assert spc_config["stage_name"] == STAGE_NAME
assert spc_config["route"] == ROUTE
assert spc_config["scope"] == SCOPE
assert spc_config["holdout_test_contact"] is HOLDOUT_TEST_CONTACT
assert spc_config["synthetic_labels"] is SYNTHETIC_LABELS
assert spc_config["train_domain_only"] is TRAIN_DOMAIN_ONLY
assert stage_inputs["stage00_run_id"] == STAGE00_RUN_ID
assert Path(stage_inputs["stage00_runtime_run_dir"]) == STAGE00_OUTPUT_DIR
assert stage_inputs["stage01_run_id"] == STAGE01_RUN_ID
assert Path(stage_inputs["stage01_runtime_run_dir"]) == STAGE01_OUTPUT_DIR
assert stage_inputs["stage02_real_run_id"] == STAGE02_REAL_RUN_ID
assert Path(stage_inputs["stage02_real_runtime_run_dir"]) == STAGE02_REAL_OUTPUT_DIR
assert Path(stage_inputs["raw_data_dir"]) == RAW_DATA_DIR
assert Path(stage_outputs["output_dir"]) == OUTPUT_DIR
assert stage_outputs["run_id"] == SPC_RUN_ID
assert Path(stage_checkpointing["checkpoint_dir"]) == CHECKPOINT_ROOT
assert stage_inputs["raw_data_manifest"] == "configs/lst_models_data.yaml"
assert spc_config["candidate"]["candidate_id"] == "price_volume_time_w20"
assert spc_config["model"]["family"] == "tcn"
assert spc_config["model"]["probe_id"] == "tcn_tiny"
assert spc_config["model"]["hpo_profile_id"] == "tcn_p01"
assert [arm["strength"] for arm in spc_config["injection"]["arms"]] == ARM_STRENGTHS
assert spc_config["injection"]["injection_seed"] == INJECTION_SEED
assert spc_config["train_inner"]["n_folds"] == 3
assert spc_config["train_inner"]["seeds"] == FROZEN_SEEDS
assert spc_config["criteria"]["minimum_positive_ticker_count"] == 3
assert spc_config["criteria"]["detection_strengths"] == [0.02, 0.05]
assert spc_config["criteria"]["monotone_strengths"] == [0.0, 0.02, 0.05]
assert spc_config["criteria"]["threshold_strength"] == 0.01
torch_defaults = spc_config["probe_training_defaults"]["torch"]
assert torch_defaults["early_stopping"] == "inner_train_chronological_tail"
assert torch_defaults["epochs"] == 64
assert torch_defaults["gradient_clip_norm"] > 0
planned_fit_rows = (
    len(spc_config["injection"]["arms"])
    * int(spc_config["train_inner"]["n_folds"])
    * len(spc_config["train_inner"]["seeds"])
)
assert planned_fit_rows <= spc_config["budget"]["max_planned_fit_rows"]

from lst_models.stages.synthetic_positive_control import _validate_config
_validate_config(spc_config)

print(json.dumps({
    "stage_name": spc_config["stage_name"],
    "evidence_status": spc_config["evidence_status"],
    "arms": [arm["arm_id"] for arm in spc_config["injection"]["arms"]],
    "planned_fit_rows": planned_fit_rows,
    "candidate_id": spc_config["candidate"]["candidate_id"],
    "profile": spc_config["model"]["hpo_profile_id"],
    "seeds": spc_config["train_inner"]["seeds"],
    "train_domain_only": spc_config["train_domain_only"],
    "holdout_test_contact": spc_config["holdout_test_contact"],
}, indent=2))


## Stage 00, Stage 01, Real Stage 02, And Raw Data Input Sync

Inputs are fetched by exact frozen run id and exact Drive file IDs only. The
real Stage 02 trial ledger is pulled solely to define the preregistered null
band (the train-inner control spread of the identical machinery); no other real
Stage 02 output is read.


In [ ]:
from lst_models.artifacts import require_artifacts

def find_unique_drive_child(service, parent_id, name, mime_type=None):
    escaped_name = quote_drive_query_value(name)
    query_parts = [f"name = '{escaped_name}'", f"'{parent_id}' in parents", "trashed = false"]
    if mime_type:
        query_parts.append(f"mimeType = '{mime_type}'")
    response = service.files().list(q=" and ".join(query_parts), spaces="drive", fields="files(id, name, mimeType, size)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) != 1:
        raise FileNotFoundError(f"expected exactly one Drive item named {name!r} under parent {parent_id}; found {len(matches)}")
    return matches[0]

def resolve_drive_folder(service, path_parts):
    folder_id = "root"
    folder_mime = "application/vnd.google-apps.folder"
    for folder_name in path_parts:
        folder = find_unique_drive_child(service, folder_id, folder_name, folder_mime)
        folder_id = folder["id"]
    return folder_id

def download_drive_file(service, file_id, output_path):
    from googleapiclient.http import MediaIoBaseDownload
    output_path.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with output_path.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

def get_drive_service_for_input_sync():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError("Drive sync only works inside Colab with Google API dependencies.") from exc
    auth.authenticate_user()
    return build("drive", "v3")

def sync_stage_artifacts_from_drive(service, output_dir, path_parts, required_names, label):
    run_folder_id = resolve_drive_folder(service, path_parts)
    for artifact_name in required_names:
        output_path = Path(output_dir) / artifact_name
        if output_path.exists():
            continue
        file_meta = find_unique_drive_child(service, run_folder_id, artifact_name)
        download_drive_file(service, file_meta["id"], output_path)
        print(f"downloaded {label}: {artifact_name}")

def sync_raw_data_from_drive(service):
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    raw_source = raw_manifest["raw_source"]
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for ticker in raw_manifest["tickers"]:
        spec = raw_source["files"][ticker]
        output_path = RAW_DATA_DIR / spec["name"]
        if output_path.exists():
            continue
        download_drive_file(service, spec["file_id"], output_path)
        print(f"downloaded raw data: {output_path.name}")

required_stage00_artifacts = spc_config["inputs"]["required_stage00_artifacts"]
required_stage01_artifacts = spc_config["inputs"]["required_stage01_artifacts"]
required_stage02_artifacts = spc_config["inputs"]["required_stage02_artifacts"]
if RUN_SPC:
    service = get_drive_service_for_input_sync()
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    missing_raw = []
    for ticker in raw_manifest["tickers"]:
        raw_path = RAW_DATA_DIR / raw_manifest["raw_source"]["files"][ticker]["name"]
        if not raw_path.exists():
            missing_raw.append(raw_path.name)
    if missing_raw and RUN_RAW_DATA_SYNC:
        print("Missing raw data files; syncing from Drive file IDs:", missing_raw)
        sync_raw_data_from_drive(service)
    stage00_missing = [name for name in required_stage00_artifacts if not (STAGE00_OUTPUT_DIR / name).exists()]
    if stage00_missing and RUN_STAGE00_DRIVE_SYNC:
        print("Missing Stage 00 artifacts; syncing exact frozen run from Drive.")
        sync_stage_artifacts_from_drive(service, STAGE00_OUTPUT_DIR, STAGE00_DRIVE_PATH_PARTS, stage00_missing, "stage00")
    stage01_missing = [name for name in required_stage01_artifacts if not (STAGE01_OUTPUT_DIR / name).exists()]
    if stage01_missing and RUN_STAGE01_DRIVE_SYNC:
        print("Missing Stage 01 artifacts; syncing exact frozen run from Drive.")
        sync_stage_artifacts_from_drive(service, STAGE01_OUTPUT_DIR, STAGE01_DRIVE_PATH_PARTS, stage01_missing, "stage01")
    stage02_missing = [name for name in required_stage02_artifacts if not (STAGE02_REAL_OUTPUT_DIR / name).exists()]
    if stage02_missing and RUN_STAGE02_REAL_DRIVE_SYNC:
        print("Missing real Stage 02 null-band artifacts; syncing exact frozen run from Drive.")
        sync_stage_artifacts_from_drive(service, STAGE02_REAL_OUTPUT_DIR, STAGE02_REAL_DRIVE_PATH_PARTS, stage02_missing, "stage02_real")
    require_artifacts(STAGE00_OUTPUT_DIR, required_stage00_artifacts)
    require_artifacts(STAGE01_OUTPUT_DIR, required_stage01_artifacts)
    require_artifacts(STAGE02_REAL_OUTPUT_DIR, required_stage02_artifacts)
    print("Stage 00, Stage 01, real Stage 02, and raw data input checks passed.")
else:
    print("RUN_SPC=False; input sync skipped.")


## Run The Synthetic Positive Control

**Reminder: TRAIN DOMAIN ONLY.** The runner re-asserts, in code, that every
event row, bar, feature row, window target, and fold boundary sits strictly
before 2013-09-16, and it fails closed otherwise. 4 arms x 3 folds x 2 seeds =
24 `tcn_tiny` fits; the runner writes a local checkpoint after every completed
arm.


In [ ]:
if RUN_SPC:
    try:
        from lst_models.stages.synthetic_positive_control import run_stage
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("Missing entry point: src/lst_models/stages/synthetic_positive_control.py.") from exc
    result = run_stage(spc_config)
    display(result)
    if Path(result.output_dir).name != SPC_RUN_ID:
        raise RuntimeError(f"run id mismatch: {Path(result.output_dir).name} != {SPC_RUN_ID}")
    if Path(result.output_dir) != SPC_OUTPUT_DIR:
        raise RuntimeError(f"output dir mismatch: {Path(result.output_dir)} != {SPC_OUTPUT_DIR}")
else:
    print("RUN_SPC=False; synthetic positive control not executed.")


## Durable Drive Result Backup

Validates the required artifact list, uploads every run file to
`My Drive/lst_models/results/v2_synthetic_positive_control/<run_id>/`, and
writes `drive_backup_manifest.json` last (its own byte size recorded as null,
self-reference).


In [ ]:
def get_drive_service_for_spc_backup():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError("Drive backup only works inside Colab with Google API dependencies.") from exc
    auth.authenticate_user()
    return build("drive", "v3")

def find_or_create_drive_folder(service, parent_id, name):
    escaped_name = quote_drive_query_value(name)
    folder_mime = "application/vnd.google-apps.folder"
    query = f"name = '{escaped_name}' and '{parent_id}' in parents and mimeType = '{folder_mime}' and trashed = false"
    response = service.files().list(q=query, spaces="drive", fields="files(id, name)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) > 1:
        raise RuntimeError(f"duplicate Drive folders named {name!r} under parent {parent_id}; resolve manually")
    if matches:
        return matches[0]["id"]
    created = service.files().create(
        body={"name": name, "mimeType": folder_mime, "parents": [parent_id]}, fields="id"
    ).execute()
    return created["id"]

def upload_or_update_drive_file(service, folder_id, local_path):
    from googleapiclient.http import MediaFileUpload
    escaped_name = quote_drive_query_value(local_path.name)
    query = f"name = '{escaped_name}' and '{folder_id}' in parents and trashed = false"
    response = service.files().list(q=query, spaces="drive", fields="files(id, name)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) > 1:
        raise RuntimeError(f"duplicate Drive files named {local_path.name!r} under folder {folder_id}; resolve manually")
    media = MediaFileUpload(str(local_path), resumable=True)
    if matches:
        updated = service.files().update(fileId=matches[0]["id"], media_body=media, fields="id").execute()
        return updated["id"]
    created = service.files().create(
        body={"name": local_path.name, "parents": [folder_id]}, media_body=media, fields="id"
    ).execute()
    return created["id"]

if RUN_SPC_DRIVE_BACKUP and RUN_SPC:
    from lst_models.stages.synthetic_positive_control import REQUIRED_SPC_ARTIFACTS
    local_run_dir = SPC_OUTPUT_DIR
    per_arm_files = sorted(path.name for path in local_run_dir.glob("spc_trials_arm_*.csv"))
    required_backup_files = list(REQUIRED_SPC_ARTIFACTS) + per_arm_files
    missing_required_artifacts = [name for name in required_backup_files if not (local_run_dir / name).exists()]
    if missing_required_artifacts:
        raise FileNotFoundError(
            "Missing required v2 synthetic positive control artifacts before Drive backup: "
            f"{missing_required_artifacts} in {local_run_dir}"
        )
    service = get_drive_service_for_spc_backup()
    parent_id = "root"
    for folder_name in SPC_DRIVE_RESULT_PATH_PARTS + [SPC_RUN_ID]:
        parent_id = find_or_create_drive_folder(service, parent_id, folder_name)
    drive_folder_id = parent_id
    uploaded_files = []
    for name in required_backup_files:
        local_path = local_run_dir / name
        drive_file_id = upload_or_update_drive_file(service, drive_folder_id, local_path)
        uploaded_files.append({
            "file_name": name,
            "relative_path": name,
            "drive_file_id": drive_file_id,
            "bytes": local_path.stat().st_size,
        })
        print("uploaded:", name)
    backup_manifest_path = local_run_dir / "drive_backup_manifest.json"
    backup_manifest = {
        "stage_name": STAGE_NAME,
        "run_id": SPC_RUN_ID,
        "local_output_dir": str(local_run_dir),
        "drive_path_parts": SPC_DRIVE_RESULT_PATH_PARTS + [SPC_RUN_ID],
        "drive_folder_id": drive_folder_id,
        "uploaded_files": uploaded_files + [{
            "file_name": "drive_backup_manifest.json",
            "relative_path": "drive_backup_manifest.json",
            "drive_file_id": None,
            "bytes": None,
            "self_reference": True,
        }],
        "sync_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "synthetic_labels": True,
        "holdout_test_contact": False,
    }
    backup_manifest_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    manifest_file_id = upload_or_update_drive_file(service, drive_folder_id, backup_manifest_path)
    backup_manifest["uploaded_files"][-1]["drive_file_id"] = manifest_file_id
    backup_manifest_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    print("stage_run_id:", SPC_RUN_ID)
    print("drive_path:", "/".join(["My Drive"] + SPC_DRIVE_RESULT_PATH_PARTS + [SPC_RUN_ID]))
    print("drive_folder_id:", drive_folder_id)
else:
    print("RUN_SPC_DRIVE_BACKUP disabled or RUN_SPC=False; Drive backup skipped.")


## Artifact Display


In [ ]:
if RUN_ARTIFACT_DISPLAY:
    import pandas as pd
    arm_summary = pd.read_csv(SPC_OUTPUT_DIR / "spc_arm_summary.csv")
    display(arm_summary)
    with (SPC_OUTPUT_DIR / "spc_criteria_readout.json").open("r", encoding="utf-8") as handle:
        criteria_readout = json.load(handle)
    print(json.dumps({key: criteria_readout[key] for key in [
        "overall_outcome", "p1_pass", "p2_monotone_pass", "p3_detection_pass",
        "threshold_arm_flags_signal", "null_band_abs", "evidence_status",
    ]}, indent=2))
    sentinel_ledger = pd.read_csv(SPC_OUTPUT_DIR / "spc_sentinel_ledger.csv")
    display(sentinel_ledger.groupby("arm_id")[
        ["observed_delta", "label_shuffle_mean", "label_shuffle_p_value", "time_reverse_delta"]
    ].mean())
else:
    print("RUN_ARTIFACT_DISPLAY=False; artifact display skipped.")


## Interpretation Guard

- Everything above is **synthetic-label protocol-validation evidence**. It says
  whether the measurement chain responds to a known planted signal; it says
  nothing about any real market edge and selects no model.
- FORBIDDEN wording in any summary built on this run: market evidence, real
  edge confirmed, model evidence, profitable, clean test, final model.
- Outcomes map to paper edits ONLY through the preregistration's section 11
  outcome map (`docs/protocols/v2_positive_control_preregistration_20260701.md`)
  and the claims-ledger process; numbers may be quoted only from
  `spc_arm_summary.csv` / `spc_criteria_readout.json` of the completed run.
- The s=0.01 arm is diagnostic (sensitivity at the scale of the recorded
  observed edge); it carries no pass/fail role.
- Any deviation from the preregistration (config edit, rerun, arm change) must
  be recorded in the preregistration's section 12 deviation log BEFORE results
  are interpreted.
